# PushT：Diffusion Policy の実行ステップ数（n_action_steps）比較

公開されている学習済み Diffusion Policy（`lerobot/diffusion_pusht`）を PushT シミュレーションで評価し、
**予測した行動を何ステップ実行してから再計画するか**（`n_action_steps`）だけを変えて、成功率と実行時間を比較します。

- モデルの再学習は行いません（推論のみ）。
- 動作確認環境：Google Colab、T4 GPU、Python 3.13、LeRobot 0.6.1
- 上から順に実行してください。セル6・7は時間がかかります（合計約80分）。

In [ ]:
# 1. GPUとPythonのバージョンを確認
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print(sys.version)

In [ ]:
# 2. インストール（LeRobot + PushT環境）
# Colab標準のtorchaoが古く、diffusersの読み込み時にImportErrorになるため更新する
!pip install -q "lerobot[pusht]==0.6.1"
!pip install -q --upgrade torchao

## 学習済みモデルの準備

Hub上の `lerobot/diffusion_pusht` は旧形式で、正規化の値が重みの中に入っており、LeRobot 0.6.1 が要求する
`policy_preprocessor.json` がありません。そこで、前処理・後処理だけを新形式に変換し、元の重みと組み合わせます。

In [ ]:
# 3. 旧形式モデルの前処理（正規化）を新形式に変換
# ※最後にTracebackが出るが想定内。前処理・後処理(policy_*)は保存済みで、
#   モデル本体の保存（config.jsonの書き出し）だけがdraccusの不具合で失敗する。
!python -m lerobot.processor.migrate_policy_normalization \
  --pretrained-path lerobot/diffusion_pusht \
  --output-dir /content/diffusion_pusht_migrated
!ls /content/diffusion_pusht_migrated

In [ ]:
# 4. 元の重みと設定に、変換済みの前処理・後処理を組み合わせる
from huggingface_hub import snapshot_download
import shutil, glob, os

MODEL_DIR = "/content/dp_local"
snapshot_download("lerobot/diffusion_pusht", local_dir=MODEL_DIR)
for f in glob.glob("/content/diffusion_pusht_migrated/policy_*"):
    shutil.copy(f, MODEL_DIR)
print(sorted(os.listdir(MODEL_DIR)))

## 実験条件

- 変えるのは `n_action_steps` だけです。シード（1000）、エピソード数、バッチサイズは全条件で同じにします。
- このモデルは16ステップ分の行動を予測し、過去2ステップの観測を使うため、`n_action_steps` の上限は 16 − 2 + 1 = 15 です。
- 値を小さくすると再計画（推論）の回数が増え、実行時間が長くなります。

In [ ]:
# 5. 評価を実行する関数（条件ごとにlerobot-evalを呼ぶ）
import subprocess, time

OUT_ROOT = "/content/outputs"
SEED = 1000
BATCH_SIZE = 10
os.makedirs(OUT_ROOT, exist_ok=True)

def run_eval(n_action_steps, n_episodes):
    """指定した実行ステップ数・エピソード数で評価し、出力フォルダを返す"""
    out_dir = f"{OUT_ROOT}/nas{n_action_steps}_ep{n_episodes}"
    cmd = [
        "lerobot-eval",
        f"--policy.path={MODEL_DIR}",
        "--env.type=pusht",
        f"--eval.n_episodes={n_episodes}",
        f"--eval.batch_size={BATCH_SIZE}",
        "--eval.use_async_envs=false",  # 非同期だとワーカー内でgym_pushtが見つからず失敗するため
        "--policy.device=cuda",
        f"--policy.n_action_steps={n_action_steps}",
        f"--seed={SEED}",
        f"--output_dir={out_dir}",
    ]
    env = dict(os.environ, SDL_VIDEODRIVER="dummy")  # 画面なし環境で描画するため
    t0 = time.time()
    with open(f"{out_dir}.log", "w") as log:
        r = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    print(f"N={n_action_steps:2d} ep={n_episodes:3d} 終了コード={r.returncode} 所要={time.time() - t0:.0f}秒", flush=True)
    return out_dir

In [ ]:
# 6. 本実験：5条件 × 30エピソード（T4で約50分）
for n in [15, 12, 8, 4, 2]:
    run_eval(n, 30)

In [ ]:
# 7. 補足実験：差の小さい15と8を100エピソードで再評価（T4で約32分）
for n in [15, 8]:
    run_eval(n, 100)

In [ ]:
# 8. 結果の集計（成功数、Wilsonの95%信頼区間、実行時間）
import json, math
import pandas as pd

def wilson(k, n, z=1.96):
    """成功率のWilson 95%信頼区間"""
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

rows = []
for path in sorted(glob.glob(f"{OUT_ROOT}/nas*_ep*/eval_info.json")):
    info = json.load(open(path))
    m, o = info["per_task"][0]["metrics"], info["overall"]
    k, n = sum(m["successes"]), len(m["successes"])
    lo, hi = wilson(k, n)
    fails = [r for r, s in zip(m["max_rewards"], m["successes"]) if not s]
    rows.append({
        "n_action_steps": int(path.split("/nas")[1].split("_")[0]),
        "episodes": n,
        "successes": k,
        "success_rate": k / n,
        "ci_low": lo,
        "ci_high": hi,
        "avg_max_reward": o["avg_max_reward"],
        "fail_avg_max_reward": sum(fails) / len(fails) if fails else float("nan"),
        "eval_s": o["eval_s"],
        "sec_per_success": o["eval_s"] / k if k else float("nan"),
    })

df = pd.DataFrame(rows).sort_values(["episodes", "n_action_steps"], ascending=[True, False])
df.to_csv(f"{OUT_ROOT}/summary.csv", index=False)
df.round(3)

In [ ]:
# 9. 条件間の差の検定（両側Fisher正確検定）
from scipy.stats import fisher_exact

def compare(n1, n2, episodes):
    a = df[(df.n_action_steps == n1) & (df.episodes == episodes)].iloc[0]
    b = df[(df.n_action_steps == n2) & (df.episodes == episodes)].iloc[0]
    table = [[a.successes, a.episodes - a.successes], [b.successes, b.episodes - b.successes]]
    _, p = fisher_exact(table)
    print(f"N={n1} vs N={n2}（{episodes}エピソード）: {a.successes}/{a.episodes} vs {b.successes}/{b.episodes}  p={p:.3f}")

compare(4, 8, 30)
compare(2, 8, 30)
compare(15, 8, 30)
compare(15, 8, 100)

In [ ]:
# 10. グラフ（成功率と実行時間）
# Colabには日本語フォントがないため、グラフのラベルは英語にしている
import matplotlib.pyplot as plt

main = df[df.episodes == 30].sort_values("n_action_steps")
x = main.n_action_steps.astype(str)
err = [(main.success_rate - main.ci_low) * 100, (main.ci_high - main.success_rate) * 100]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(x, main.success_rate * 100, yerr=err, capsize=4)
axes[0].set_xlabel("n_action_steps")
axes[0].set_ylabel("Success rate (%)")
axes[0].set_title("Success rate (30 episodes, 95% CI)")
axes[1].bar(x, main.eval_s)
axes[1].set_xlabel("n_action_steps")
axes[1].set_ylabel("Evaluation time (s)")
axes[1].set_title("Evaluation time (30 episodes)")
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/results.png", dpi=150)
plt.show()

In [ ]:
# 11. 動画に条件・番号・成否の名前を付けて集める
# ※lerobot-evalは各条件の最初の10エピソード分だけ動画を保存する
from IPython.display import Video, display

SEL = "/content/selected_videos"
os.makedirs(SEL, exist_ok=True)
for path in sorted(glob.glob(f"{OUT_ROOT}/nas*_ep30/eval_info.json")):
    tag = "N" + path.split("/nas")[1].split("_")[0]  # 例: N8
    m = json.load(open(path))["per_task"][0]["metrics"]
    for i, (ok, video) in enumerate(zip(m["successes"], m["video_paths"])):
        shutil.copy(video, f"{SEL}/{tag}_ep{i:02d}_{'success' if ok else 'fail'}.mp4")
print(len(os.listdir(SEL)), "本")

# 例：同じ初期状態（エピソード2）で N=8 と N=4 を並べて表示
for name in ["N8_ep02", "N4_ep02"]:
    f = glob.glob(f"{SEL}/{name}_*.mp4")[0]
    print(os.path.basename(f))
    display(Video(f, embed=True, width=300))

In [ ]:
# 12. 結果をPCに保存（Colabはランタイムが切れると/contentが消えるため）
!cd /content && zip -rq results.zip outputs selected_videos
from google.colab import files
files.download("/content/results.zip")